### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import combinations
from astropy.constants import si as constants
from astropy import units as u
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm 
from survey_tools import sky
from tabulate import tabulate

### Options

In [ ]:
R = 3000  # 3000 or 8000
d = 0.05  # spaxel size in arcsec
n = 2*2*2 # aperture size in pixels (2x2 spaxels x2 spectral bins)

lines = [
    {'name': 'Ha'  , 'wavelength_vac': np.array([0.6564610           ]), 'sigma': 200},
    {'name': 'NII' , 'wavelength_vac': np.array([0.6549890, 0.6585270]), 'sigma': 200},
    {'name': 'Hb'  , 'wavelength_vac': np.array([0.4862680           ]), 'sigma': 200},
    {'name': 'OIII', 'wavelength_vac': np.array([0.4960295, 0.5008240]), 'sigma': 200},
]

airmass = 1.5 # 1.0 or 1.5 or 2.0

max_time           = 4*3600 # integration time
min_SNR            = 3      # minimum SNR
min_sky_trans      = 0.80   # percent
sky_trans_multiple = 0.5    # multiple of FWHM
sky_line_multiple  = 0.5    # multiple of FWHM

### Prepare

In [ ]:
match R:
    case 3000:
        bands = [
            {'name': 'YJ', 'start': 0.95, 'end': 1.35},
            {'name': 'JH', 'start': 1.25, 'end': 1.80},
            {'name': 'HK', 'start': 1.63, 'end': 2.35},
        ]
    case 8000:
        bands = [
            {'name': 'Js', 'start': 1.194, 'end': 1.350},
            {'name': 'Hs', 'start': 1.500, 'end': 1.706},
            {'name': 'Ks', 'start': 2.110, 'end': 2.379},
        ]

band_wavelength_range  = np.array([[b['start'], b['end']] for b in bands])
band_wavelength_min = np.min(band_wavelength_range)
band_wavelength_max = np.max(band_wavelength_range)
print(f"Bands: {band_wavelength_min}-{band_wavelength_max} μm (R={R})")

sky_transmission_data = sky.load_transmission_data('MaunaKea', airmass)
print(f"Sky Transmission: {sky_transmission_data['wavelength'][0]/10:.0f}-{sky_transmission_data['wavelength'][-1]/10:.0f} nm, N={len(sky_transmission_data)}, dλ = {(sky_transmission_data['wavelength'][1]-sky_transmission_data['wavelength'][0])/10:.2f} nm")

sky_background_data = sky.load_background_data('MaunaKea', airmass)
print(f"Sky Background: {sky_background_data['wavelength'][0]/10:.0f}-{sky_background_data['wavelength'][-1]/10:.0f} nm, N={len(sky_background_data)}, dλ = {(sky_background_data['wavelength'][1]-sky_background_data['wavelength'][0])/10:.2f} nm")

### Calculations

In [ ]:
for l in lines:
    l['wavelength'] = sky.get_vacuum_to_air_wavelength(l['wavelength_vac']*u.micron).value

In [ ]:
# Reference Source
source_w     = 1.6e3   # reference wavelength in nm
source_flux  = 1e-16   # erg/s/cm^2
source_A     = 1.0     # arcsec^2
source_sigma = 200     # velocity dispersion in km/s

# Gemini North and GIRMOS
D            = 7.9     # telescope diameter in meters
epsilon      = 1.3/7.9 # pupil obscuration
throughput   = 0.23    # total system throughput

# Derived quantities
Aeff = np.pi * (D/2 * (1-epsilon))**2
source_fwhm = source_w * 2.35482 * source_sigma*1e3 / constants.c.value
S = source_flux
S *= 1e-7*1e4      # convert erg/s/cm^2 to J/s/m^2
S /= constants.h.value * constants.c.value / (source_w*1e-9) # convert from J to ph
S /= source_A      # per arcesec^2
S /= source_fwhm   # per nm
S *= min_sky_trans # after passing through the atmosphere
B = S**2 * Aeff * throughput * d**2 * source_w/R * n * max_time / min_SNR**2 - S # Assume sky limited: SNR = S/sqrt(S+B)

# Minimum sky photon rate
min_sky_rate = np.round(B,1)
print(f"Skyline Flux > {min_sky_rate} ph/s/m^2/arcsec^2/nm")

In [ ]:
max_line_wavelength = np.max([np.max(l['wavelength']) for l in lines])
min_line_wavelength = np.min([np.min(l['wavelength']) for l in lines])

dlambda = np.round(band_wavelength_min / R, 5)
N = int((band_wavelength_max - band_wavelength_min) / dlambda)

min_redshift = np.round(band_wavelength_min / min_line_wavelength - 1, 5)
max_redshift = np.round(band_wavelength_max / max_line_wavelength  - 1, 5)
dz = np.round((max_redshift - min_redshift) / N, 5)
max_redshift = min_redshift + dz * (N-1) # Adjust max_redshift so N dz's fit in range
redshifts = np.linspace(min_redshift, max_redshift, N)
print(f"Redshift Range to Search: {min_redshift:.3f}-{max_redshift:.3f}, N={N}, dz = {dz:.5f}")

In [ ]:
def plot_redshift_spectrum(redshift, sky_background_data, line, R, min_sky_rate, rejected=False, plot_log=False):
    wavelength_Ha_atm = line['wavelength'] * (1 + redshift) * 1e4
    avoid_multiple = 0.5
    ph_rate_Ha = 0.3
    FWHM_Ha = 2.5 * 10
    dwavelength    = wavelength_Ha_atm / R
    dwavelength_Ha = np.sqrt(dwavelength**2 + FWHM_Ha**2)
    sigma_Ha   = dwavelength_Ha / 2.35482 # FWHM -> sigma

    plot_wavelength_range = [wavelength_Ha_atm-25, wavelength_Ha_atm+25]

    lgray = (0.5,0.5,0.5)
    alpha = 0.3
    wavelengths = np.linspace(plot_wavelength_range[0], plot_wavelength_range[1], 1000)
    sky_background_data_low_res = sky.get_low_res_background(sky_background_data, plot_wavelength_range, R)
    sky_lines = sky.find_sky_lines(sky_background_data_low_res, min_sky_rate)

    ylim = [1e-1, max(sky_background_data_low_res['emission'])*1.1]

    wavelength = sky_background_data_low_res['wavelength']
    _, ax = plt.subplots(figsize=(10, 5))
    ax.plot(wavelength, sky_background_data_low_res['emission'], linestyle='-', color='b', linewidth=1)
    ax.scatter(sky_lines['wavelength'], sky_lines['emission'], marker='x', color='b')
    if len(sky_lines) <= 5:
        plt.hlines(y=sky_lines["width_height"], xmin=sky_lines["wavelength_low"], xmax=sky_lines["wavelength_high"], linestyle='-', color='b')        
    ax.axhline(min_sky_rate, linestyle=':', linewidth=1, color='k')
    ax.set_xlabel('Wavelength [Angstrom]')
    ax.set_ylabel('Emission [$ph/s/m^2/arcsec^2/nm$]')
    ax.set_xlim(plot_wavelength_range)
    if plot_log:
        ax.set_yscale('log')
        ax.set_ylim(ylim)

    colour = 'r' if rejected else 'g'
    ax.fill_between(wavelength, ylim[0], ylim[1], where=(abs(wavelength - wavelength_Ha_atm) < dwavelength_Ha*avoid_multiple), facecolor=lgray, alpha=alpha)
    ax.plot(wavelengths, ph_rate_Ha * np.sqrt(2*np.pi) * sigma_Ha * norm.pdf(wavelengths, wavelength_Ha_atm, sigma_Ha), linestyle='-', linewidth=2, color=colour)
    #plt.hlines(y=PF_Ha/2, xmin=(lambda_Ha_atm-dLambda_Ha/2), xmax=(lambda_Ha_atm+dLambda_Ha/2), linestyle='-', linewidth=2, color=colour)        


In [ ]:
usable_redshifts = np.ones_like(redshifts, dtype=bool)
redshift_num_bands = np.zeros_like(redshifts, dtype=int)

plotted = False

for i, z in enumerate(redshifts):
    line_bands = np.zeros((len(lines), len(bands)), dtype=np.bool)

    for j, l in enumerate(lines):
        w = l['wavelength'] * (1 + z)
        fwhm = np.sqrt((w/R)**2 + (w * 2.35482 * l['sigma'] / constants.c.to('km/s').value)**2)

        reject = sky.reject_emission_line(
            sky_background_data,
            sky_transmission_data,
            w*1e4, # convert micron to angstrom
            fwhm*1e4, # convert micron to angstrom
            R,
            allowed_wavelength_range=band_wavelength_range*1e4, # convert micron to angstrom
            trans_minimum=min_sky_trans, 
            trans_dLambda_multiple=sky_trans_multiple, 
            avoid_dLambda_multiple=sky_line_multiple, 
            min_photon_rate=min_sky_rate
        )

        if not isinstance(reject, np.ndarray):
            reject = np.array([reject])

        if np.all(reject):
            usable_redshifts[i] = False
            break
        else:
            for k in range(len(w)):
                if not reject[k]:
                    band_indexes = np.where((band_wavelength_range[:,0] <= w[k]) & (w[k] <= band_wavelength_range[:,1]))[0]
                    line_bands[j, band_indexes] = True
    
    if usable_redshifts[i]:
        min_bands = np.inf
        for num_cols in range(1, line_bands.shape[1] + 1):
            for band_indices in combinations(range(line_bands.shape[1]), num_cols):
                if np.all(np.any(line_bands[:, band_indices], axis=1)):
                    min_bands = num_cols
                    break
            if min_bands != np.inf:
                break
        redshift_num_bands[i] = min_bands if min_bands != np.inf else 0

        if not plotted:
            plot_redshift_spectrum(z, sky_background_data, lines[0], R, min_sky_rate, rejected=not usable_redshifts[i], plot_log=True)          
            plotted = True

In [ ]:
print(f"Usable Redshifts: N={np.sum(usable_redshifts)}/{len(redshifts)} ({np.sum(usable_redshifts)/N:.1%})")
print("Usable Redshifts by Num Bands Required:")
for i in range(len(bands)):
    print(f"  {i+1}: {np.sum(redshift_num_bands == i+1)}")

plt.figure(figsize=(8, 4))

for i in range(len(bands)):
    if np.sum(redshift_num_bands == i+1) > 0:
        match i+1:
            case 1:
                color = 'b'
            case 2:
                color = 'g'
            case 3:
                color = 'r'
        Z = redshifts[usable_redshifts & (redshift_num_bands == i+1)]
        plt.plot(Z, np.ones_like(Z), '|', markersize=10, color=color, label=f'{i+1} band{"s" if i+1>1 else ""} (N={len(Z)})')

plt.legend()
plt.xlabel('Redshift')
plt.yticks([])
plt.title(f"Usable Redshifts for BPT with GIRMOS (R={R})")
#plt.xlim([1.5, 1.6])
plt.show()

In [ ]:
redshift_summary = Table({
    'Redshift': np.round(redshifts[usable_redshifts],6),
    'NumBands': redshift_num_bands[usable_redshifts]
})

redshift_summary.write('../output/bpt_redshifts.txt', format='ascii', overwrite=True)

display(tabulate(redshift_summary, headers=redshift_summary.colnames, tablefmt='html'))